# 🌴 Minimal Bali NbS (Canopy) Notebook

A simple, **single-notebook** workflow (no QGIS) to compute a Nature-based Solutions (NbS) score for **Bali** using:
- **Canopy Height Map (CHM)** from the Meta/WRI paper (GeoTIFF, ~1 m)
- **ESA WorldCover 10 m** (tree cover mask)
- **MODIS (or Landsat) LST** as a heat proxy
- **WorldPop 100 m** population (optional, for equity)
- **JRC Global Surface Water occurrence** (optional, for riparian/coastal canopy)

👉 Edit the file paths below and run cells top-to-bottom.


In [ ]:
# (Optional) Install dependencies in this environment
# !pip install geopandas rasterio rioxarray xarray numpy pandas shapely pyproj matplotlib scipy


In [ ]:
# ---- CONFIG: set your local file paths ----
from pathlib import Path

DATA = Path("../data")  # adjust if needed
OUT  = Path("../outputs"); OUT.mkdir(parents=True, exist_ok=True)

paths = {
    # REQUIRED
    "chm": DATA / "chm/chm_bali.tif",                     # Canopy Height Map (clipped to Bali)
    "worldcover": DATA / "worldcover/WorldCover_IDN.tif", # ESA WorldCover (Indonesia or Bali)
    "modis_lst": DATA / "lst/MOD11A2_Bali_day.tif",       # MODIS daytime LST composite (~1 km)
    # OPTIONAL (comment out if not available)
    "worldpop": DATA / "worldpop/IDN_ppp_2020_100m.tif",
    "jrc_water": DATA / "jrc/occurrence.tif",
}

params = {
    "canopy_min_height_m": 3,   # presence threshold
    "canopy_tall_height_m": 5,  # tall canopy threshold
    "buffer_water_m": 50,       # riparian/coastal buffer (requires pixel size)
    "weights": {"mitigation": 0.4, "heat": 0.3, "flood": 0.3},
}

print(paths)

In [ ]:
# ---- Imports ----
import numpy as np
import xarray as xr
import rioxarray as rxr
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import binary_dilation

def reproject_match(src_da: xr.DataArray, ref_da: xr.DataArray, resampling=Resampling.nearest):
    return src_da.rio.reproject_match(ref_da, resampling=resampling)

def norm01(da: xr.DataArray):
    vmin = np.nanpercentile(da.values, 2)
    vmax = np.nanpercentile(da.values, 98)
    out = (da - vmin) / (vmax - vmin + 1e-9)
    return out.clip(0, 1)

def zscore(da: xr.DataArray):
    m = np.nanmean(da.values); s = np.nanstd(da.values)
    return (da - m) / s if s > 0 else da * np.nan


In [ ]:
# ---- Load rasters ----
chm = rxr.open_rasterio(paths["chm"], masked=True).squeeze()         # expected high-res (1 m)
wc  = rxr.open_rasterio(paths["worldcover"], masked=True).squeeze()  # 10 m
lst = rxr.open_rasterio(paths["modis_lst"], masked=True).squeeze()   # ~1 km

wp = None; jrc = None
if paths.get("worldpop") and Path(paths["worldpop"]).exists():
    wp = rxr.open_rasterio(paths["worldpop"], masked=True).squeeze()
if paths.get("jrc_water") and Path(paths["jrc_water"]).exists():
    jrc = rxr.open_rasterio(paths["jrc_water"], masked=True).squeeze()

chm, wc, lst

In [ ]:
# ---- Align to CHM grid ----
wc  = reproject_match(wc, chm)
lst = reproject_match(lst, chm, Resampling.bilinear)
if wp is not None:
    wp = reproject_match(wp, chm, Resampling.bilinear)
if jrc is not None:
    jrc = reproject_match(jrc, chm, Resampling.bilinear)
print("Aligned rasters to CHM grid.")

In [ ]:
# ---- Masks & basic metrics ----
# Vegetation mask: ESA WorldCover class 10 = Tree cover
veg_mask = (wc == 10).astype("float32")
veg_mask = veg_mask.where(~np.isnan(chm), np.nan)

canopy_min = (chm >= params["canopy_min_height_m"]).astype("float32") * veg_mask
canopy_tall = (chm >= params["canopy_tall_height_m"]).astype("float32") * veg_mask

canopy_height = chm.where(veg_mask == 1)
canopy_volume = (chm * canopy_min).astype("float32")  # simple mitigation proxy

# Cooling index: higher when tall canopy coincides with lower LST
lst_z = zscore(lst)
chm_z = zscore(canopy_height)
cooling_idx = (-1.0 * lst_z) * (chm_z)

# Riparian/coastal canopy (optional, requires jrc)
riparian_canopy = None
if jrc is not None:
    water_mask = (jrc >= 50).astype("float32").fillna(0)
    res_x = abs(chm.rio.resolution()[0])
    px = max(1, int(params["buffer_water_m"] / res_x))
    buf = binary_dilation(water_mask.values.astype(bool), iterations=px)
    riparian_zone = xr.DataArray(buf, coords=water_mask.coords, dims=water_mask.dims)
    riparian_canopy = canopy_min * riparian_zone

# Equity metric (optional, requires worldpop)
canopy_per_capita = None
if wp is not None:
    res_x, res_y = map(abs, chm.rio.resolution())
    pixel_area = res_x * res_y
    canopy_area_m2 = canopy_min * pixel_area
    canopy_per_capita = canopy_area_m2 / (wp + 1e-6)

print("Computed core indicators.")

In [ ]:
# ---- Normalize & combine to NbS score ----
mit_norm  = norm01(canopy_volume)
heat_norm = norm01(cooling_idx)
flood_norm = None
if riparian_canopy is not None:
    flood_norm = norm01(riparian_canopy)
else:
    # if JRC not provided, fallback to zeros to keep shapes
    flood_norm = xr.zeros_like(mit_norm)

w = params["weights"]
nbs_score = (w["mitigation"] * mit_norm +
             w["heat"] * heat_norm +
             w["flood"] * flood_norm)

print("Combined NbS score.")

In [ ]:
# ---- Quick plots ----
for title, da in [
    ("Canopy height (m)", canopy_height),
    ("Cooling index", cooling_idx),
    ("NbS score", nbs_score),
]:
    fig, ax = plt.subplots(figsize=(6,6))
    im = ax.imshow(da, origin="upper")
    ax.set_title(title)
    ax.axis("off")
    fig.colorbar(im, ax=ax, shrink=0.7)
    plt.show()


In [ ]:
# ---- Save outputs (GeoTIFF) ----
prefix = "bali_minimal"
canopy_height.rio.to_raster(OUT / f"{prefix}_canopy_height.tif", compress="lzw")
canopy_volume.rio.to_raster(OUT / f"{prefix}_canopy_volume.tif", compress="lzw")
cooling_idx.rio.to_raster(OUT / f"{prefix}_cooling_index.tif", compress="lzw")
nbs_score.rio.to_raster(OUT / f"{prefix}_nbs_score.tif", compress="lzw")
if riparian_canopy is not None:
    riparian_canopy.rio.to_raster(OUT / f"{prefix}_riparian_canopy.tif", compress="lzw")
if canopy_per_capita is not None:
    canopy_per_capita.rio.to_raster(OUT / f"{prefix}_canopy_per_capita.tif", compress="lzw")
print("Saved GeoTIFF outputs in:", OUT)
